# 🚦 SignSense: GPU Training on Google Colab

This notebook trains all 4 models for **SignSense (Traffic Sign Recognition on GTSRB - 43 classes)**:
- **M1 (Data & Custom CNN):** VGG-style CNN built from scratch (`cnn`) — 15 epochs
- **M2 (Transfer Learning):** ResNet-18 pretrained (`resnet18`) — 8 epochs
- **M2 (Edge Deployment):** MobileNetV3-Small (`mobilenet`) — 10 epochs
- **M3 (Robustness & Defence):** Custom CNN with FGSM Adversarial Training (`cnn_robust`) — 15 epochs, eps=0.02

**Hardware requirement:** Select **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Clone repository
!rm -rf SignSense-Traffic-Sign-Recognition
!git clone https://github.com/Madhura1705/SignSense-Traffic-Sign-Recognition.git
%cd SignSense-Traffic-Sign-Recognition

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt kagglehub

In [ ]:
# 4. Prepare GTSRB dataset (downloads via kagglehub and sorts class folders)
!python scripts/prepare_kaggle.py

### 🏋️ Train All 4 Models
Results are saved into `models/` and evaluation outputs merge into `outputs/metrics.json`.

In [ ]:
# M1: Train Custom CNN (target: >= 95% test accuracy)
!python train.py --source folder --data data/gtsrb --models cnn --epochs-cnn 15 --workers 2

In [ ]:
# M2: Train ResNet-18 (target: >= 97% test accuracy)
!python train.py --source folder --data data/gtsrb --models resnet18 --epochs-resnet 8 --workers 2

In [ ]:
# M2: Train MobileNetV3-Small (target: >= 94% test accuracy)
!python train.py --source folder --data data/gtsrb --models mobilenet --epochs-resnet 10 --workers 2

In [ ]:
# M3: Train Robust CNN with Adversarial Training (FGSM eps=0.02)
!python train.py --source folder --data data/gtsrb --models cnn_robust --epochs-cnn 15 --adv-eps 0.02 --workers 2

### ⚡ Edge Deployment: ONNX Export & CPU Benchmark

In [ ]:
# Run ONNX export & benchmark
!python export_onnx.py

In [ ]:
# Display comparison table
import pandas as pd
print('=== MODEL PERFORMANCE ON TEST SET ===')
display(pd.read_csv('outputs/comparison.csv'))
print('\n=== ONNX BENCHMARK (EDGE DEPLOYMENT) ===')
display(pd.read_csv('outputs/onnx_benchmark.csv'))

### 📦 Package & Download Results
This will create `signsense_results.zip` containing `models/*.pt`, `outputs/`, and `samples/`, then automatically prompt your browser to download it.

In [ ]:
import os, zipfile, glob
from google.colab import files

zip_filename = 'signsense_results.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Include all trained PyTorch weights
    for pt in glob.glob('models/*.pt'):
        zipf.write(pt)
        print(f'Packed: {pt}')
    # Include all output plots, reports and metrics
    for root, _, files_list in os.walk('outputs'):
        for f in files_list:
            p = os.path.join(root, f)
            zipf.write(p)
    # Include sample images
    for root, _, files_list in os.walk('samples'):
        for f in files_list:
            p = os.path.join(root, f)
            zipf.write(p)

size_mb = os.path.getsize(zip_filename) / 1e6
print(f'\nSuccessfully created {zip_filename} ({size_mb:.1f} MB)')
files.download(zip_filename)